In [ ]:
!pip install -U transformers accelerate
!pip install -q transformers datasets accelerate peft bitsandbytes
from huggingface_hub import login
login()
from transformers import AutoTokenizer, AutoModelForSequenceClassification
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
import pandas as pd
from datasets import Dataset
train_df = pd.read_csv(“/content/train.csv”)
test_df = pd.read_csv(“/content/test.csv”)
print(train_df.columns)
train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)
def tokenize(batch):
    return tokenizer(batch[“full_name”], padding=“max_length”, truncation=True, max_length=256)
train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)
train_ds = train_ds.rename_column(“label”, “labels”)
test_ds = test_ds.rename_column(“label”, “labels”)
train_ds.set_format(type=“torch”, columns=[“input_ids”, “attention_mask”, “labels”])
test_ds.set_format(type=“torch”, columns=[“input_ids”, “attention_mask”, “labels”])
from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(
    output_dir=“./deepseek-classifier”,
    evaluation_strategy=“epoch”,
    save_strategy=“epoch”,
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir=“./logs”,
    load_best_model_at_end=True,
    fp16=True
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds
)
trainer.train()
